# Payment Reliability Dashboard

KPI -> Question/Hypothesis -> Graph -> Conclusion

Primary data: cleaned UPI transactions.
Supporting data: aggregated reliability summary.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display
from scipy import stats

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve()
RAW_DATA_PATH = PROJECT_ROOT / "data/processed/cleaned_upi_transactions_2024.csv"
AGG_DATA_PATH = PROJECT_ROOT / "data/processed/reliability_aggregated.csv"

raw_df = pd.read_csv(RAW_DATA_PATH)
agg_df = pd.read_csv(AGG_DATA_PATH)

if "timestamp" in raw_df.columns:
    raw_df["timestamp"] = pd.to_datetime(raw_df["timestamp"], errors="coerce")

for column in ["amount_inr", "hour_of_day", "is_peak_hour", "failure_flag", "is_outlier_amount"]:
    if column in raw_df.columns:
        raw_df[column] = pd.to_numeric(raw_df[column], errors="coerce")

for column in ["is_peak_hour", "day_of_week", "sender_bank", "receiver_bank", "device_type", "network_type", "transaction_status", "sender_state", "amount_category", "value_bucket"]:
    if column in raw_df.columns:
        raw_df[column] = raw_df[column].astype(str).str.strip().str.lower()

for column in ["is_peak_hour", "day_of_week", "sender_bank", "device_type", "network_type"]:
    if column in agg_df.columns:
        agg_df[column] = agg_df[column].astype(str).str.strip().str.lower()

for column in ["total_transactions", "failed_transactions", "failure_rate", "avg_amount"]:
    if column in agg_df.columns:
        agg_df[column] = pd.to_numeric(agg_df[column], errors="coerce")

if "day_of_week" in raw_df.columns:
    day_order = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
    raw_df["day_of_week"] = pd.Categorical(raw_df["day_of_week"], categories=day_order, ordered=True)

if "day_of_week" in agg_df.columns:
    day_order = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
    agg_df["day_of_week"] = pd.Categorical(agg_df["day_of_week"], categories=day_order, ordered=True)

def raw_failure_mask(data_frame):
    if "failure_flag" in data_frame.columns:
        return pd.to_numeric(data_frame["failure_flag"], errors="coerce").fillna(0).astype(int).eq(1)
    if "transaction_status" in data_frame.columns:
        return data_frame["transaction_status"].astype(str).str.lower().str.strip().isin(["failed", "failure", "declined"])
    return pd.Series(False, index=data_frame.index)

raw_df["is_failed"] = raw_failure_mask(raw_df)

def summarize_raw(data_frame, group_col):
    temp = data_frame.copy()
    temp["_failed"] = raw_failure_mask(temp)
    summary = temp.groupby(group_col, dropna=False).agg(
        total_transactions=("_failed", "size"),
        failed_transactions=("_failed", "sum"),
    ).reset_index()
    summary["failure_rate"] = np.where(
        summary["total_transactions"] > 0,
        summary["failed_transactions"] / summary["total_transactions"] * 100,
        np.nan,
    )
    return summary

def summarize_agg(data_frame, group_col):
    summary = data_frame.groupby(group_col, dropna=False).agg(
        total_transactions=("total_transactions", "sum"),
        failed_transactions=("failed_transactions", "sum"),
    ).reset_index()
    summary["failure_rate"] = np.where(
        summary["total_transactions"] > 0,
        summary["failed_transactions"] / summary["total_transactions"] * 100,
        np.nan,
    )
    return summary

def contingency_from_raw(data_frame, group_col):
    temp = data_frame.copy()
    temp["_status"] = np.where(raw_failure_mask(temp), "failed", "success")
    table = pd.crosstab(temp[group_col], temp["_status"])
    return table

def contingency_from_agg(data_frame, group_col):
    table = data_frame.groupby(group_col, dropna=False).agg(
        failed=("failed_transactions", "sum"),
        success=("total_transactions", lambda values: values.sum())
    )
    table["success"] = table["success"] - table["failed"]
    return table[["failed", "success"]]

def chi_square_test(table):
    if table.shape[0] < 2 or table.shape[1] < 2:
        return np.nan, np.nan
    chi2, p_value, _, _ = stats.chi2_contingency(table)
    return chi2, p_value

def weighted_rate(total_transactions, failed_transactions):
    total_transactions = np.asarray(total_transactions, dtype=float)
    failed_transactions = np.asarray(failed_transactions, dtype=float)
    total = total_transactions.sum()
    failed = failed_transactions.sum()
    return np.nan if total == 0 else failed / total * 100

def top_n_summary(summary, n=5):
    cols = [col for col in summary.columns if col not in {"total_transactions", "failed_transactions", "failure_rate"}]
    sort_cols = ["failure_rate", "total_transactions"]
    return summary.sort_values(sort_cols, ascending=[False, False]).head(n)[cols + ["total_transactions", "failed_transactions", "failure_rate"]]

px.defaults.template = "plotly_white"
pd.set_option("display.max_columns", None)

## Data Sources Overview

### Raw Data (`cleaned_upi_transactions_2024.csv`)
- **Granularity:** Transaction-level detail (250,000 individual transactions)
- **Contains:** Timestamp, sender/receiver details, device type, network type, geographic location (state), transaction amount, and failure indicators
- **Use Case:** Enables hour-by-hour, state-by-state, and bank-pair analysis with full temporal precision
- **Strength:** High detail reveals patterns across time, geography, and specific combinations that aggregated data loses

### Aggregated Data (`reliability_aggregated.csv`)
- **Granularity:** Pre-grouped summaries by bank, device, network, peak-hour flag, and day of week
- **Contains:** Total transaction counts, failed counts, and pre-computed failure rates for each group
- **Use Case:** Fast cross-checks and validation; avoids recalculating the same totals from raw data
- **Strength:** Pre-computed rates confirm whether raw calculations are accurate and complete

### Key Difference When Both Are Shown
- **Raw graphs** show the actual transaction-level failure rate for each category (most accurate, highest detail)
- **Aggregated graphs** show the same category but with pre-grouped totals (may smooth over intra-group detail but validates consistency)
- If raw and aggregated match closely, it confirms data integrity; large differences indicate potential data quality issues or aggregation scope differences

## 1. Core KPI Layer

**Question:** What is the overall reliability of the system?

**Hypothesis:** The transaction system should show a stable baseline success rate, with any sustained increase in failure rate indicating a reliability regression.

**Raw vs Aggregated Comparison:**
- **Raw KPI:** Calculated directly from all 250,000 individual transactions in the cleaned dataset
- **Aggregated KPI:** Pre-computed from the summary table (totals already summed across all group combinations)
- Both should show the same overall failure rate; differences indicate data loading or calculation discrepancies

In [ ]:
raw_total = len(raw_df)
raw_failed = int(raw_df["is_failed"].sum())
raw_success_rate = 100 * (raw_total - raw_failed) / raw_total if raw_total else np.nan
raw_failure_rate = 100 * raw_failed / raw_total if raw_total else np.nan

agg_total = int(agg_df["total_transactions"].sum()) if "total_transactions" in agg_df.columns else np.nan
agg_failed = int(agg_df["failed_transactions"].sum()) if "failed_transactions" in agg_df.columns else np.nan
agg_success_rate = 100 * (agg_total - agg_failed) / agg_total if agg_total else np.nan
agg_failure_rate = 100 * agg_failed / agg_total if agg_total else np.nan

kpi_df = pd.DataFrame([
    {"source": "raw", "total_transactions": raw_total, "failed_transactions": raw_failed, "success_rate": raw_success_rate, "failure_rate": raw_failure_rate},
    {"source": "aggregated", "total_transactions": agg_total, "failed_transactions": agg_failed, "success_rate": agg_success_rate, "failure_rate": agg_failure_rate},
])

fig_kpi = go.Figure(data=[go.Table(
    header=dict(values=list(kpi_df.columns), align="left"),
    cells=dict(values=[kpi_df[col] for col in kpi_df.columns], align="left"),
)])
fig_kpi.update_layout(title="Core KPI Comparison: Raw vs Aggregated")
fig_kpi.show()

daily = raw_df.dropna(subset=["timestamp"]).copy()
daily["txn_date"] = daily["timestamp"].dt.date
daily_summary = daily.groupby("txn_date", as_index=False).agg(
    total_transactions=("is_failed", "size"),
    failed_transactions=("is_failed", "sum"),
)
daily_summary["failure_rate"] = np.where(
    daily_summary["total_transactions"] > 0,
    daily_summary["failed_transactions"] / daily_summary["total_transactions"] * 100,
    np.nan,
 )
fig_trend = px.line(daily_summary, x="txn_date", y="failure_rate", title="Raw Daily Failure Rate Over Time")
fig_trend.update_yaxes(title="Failure Rate (%)")
fig_trend.show()

agg_daily = agg_df.groupby("day_of_week", as_index=False).agg(
    total_transactions=("total_transactions", "sum"),
    failed_transactions=("failed_transactions", "sum"),
) if "day_of_week" in agg_df.columns else pd.DataFrame()
if len(agg_daily):
    agg_daily["failure_rate"] = np.where(agg_daily["total_transactions"] > 0, agg_daily["failed_transactions"] / agg_daily["total_transactions"] * 100, np.nan)
    fig_agg_daily = px.bar(agg_daily, x="day_of_week", y="failure_rate", title="Aggregated Failure Rate by Day of Week")
    fig_agg_daily.update_yaxes(title="Failure Rate (%)")
    fig_agg_daily.show()

**Conclusions:**
1. The system maintains a consistent baseline failure rate of ~5% with intermittent spikes up to 7–8%, indicating underlying instability driven by transient issues rather than uniform performance degradation.

2. The absence of variation across days of the week suggests failures are not demand-driven but stem from systemic factors such as infrastructure, network reliability, or bank integrations.

3. The persistence of failure rates over time with no improving trend indicates structural inefficiencies, resulting in ~12,000+ failed transactions at scale, with significant impact on user experience and platform reliability.


## 2. Time-Based Reliability

**Question:** When do failures spike?

**Hypothesis:** Failure rates will be higher during peak hours and may vary by weekday because demand and operational load are not uniform.

**Raw vs Aggregated Comparison:**
- **Raw charts (Hour, Day, Peak):** Show failure rates grouped directly from transaction timestamps; captures every hour and day with actual transaction detail
- **Aggregated chart (Day of Week):** Shows pre-aggregated failure rates by day; useful for cross-validation that raw daily data is consistent
- If raw peak-hour rate diverges from aggregated, investigate whether peak-hour definition differs or if transaction-level timestamps are miscoded

In [ ]:
hour_summary = summarize_raw(raw_df, "hour_of_day") if "hour_of_day" in raw_df.columns else pd.DataFrame()
day_summary = summarize_raw(raw_df, "day_of_week") if "day_of_week" in raw_df.columns else pd.DataFrame()

fig_hour = px.bar(hour_summary, x="hour_of_day", y="failure_rate", title="Raw Failure Rate by Hour") if len(hour_summary) else None
if fig_hour is not None:
    fig_hour.update_yaxes(title="Failure Rate (%)")
    fig_hour.show()

fig_day = px.bar(day_summary, x="day_of_week", y="failure_rate", title="Raw Failure Rate by Day of Week") if len(day_summary) else None
if fig_day is not None:
    fig_day.update_yaxes(title="Failure Rate (%)")
    fig_day.show()

peak_summary = summarize_raw(raw_df, "is_peak_hour") if "is_peak_hour" in raw_df.columns else pd.DataFrame()
if len(peak_summary):
    fig_peak = px.bar(peak_summary, x="is_peak_hour", y="failure_rate", title="Failure Rate: Peak vs Off-Peak")
    fig_peak.update_yaxes(title="Failure Rate (%)")
    fig_peak.show()

if "is_peak_hour" in raw_df.columns:
    contingency = contingency_from_raw(raw_df, "is_peak_hour")
    chi2, p_value = chi_square_test(contingency)
    print(f"Chi-square p-value for peak-hour failure difference: {p_value:.6f}")

agg_day_summary = summarize_agg(agg_df, "day_of_week") if "day_of_week" in agg_df.columns else pd.DataFrame()
if len(agg_day_summary):
    fig_agg_day = px.bar(agg_day_summary, x="day_of_week", y="failure_rate", title="Aggregated Failure Rate by Day of Week")
    fig_agg_day.update_yaxes(title="Failure Rate (%)")
    fig_agg_day.show()

Chi-square p-value for peak-hour failure difference: 0.592298


**Conclusion:** 
1. Failure rates show no statistically significant difference between peak and off-peak hours (p ≈ 0.59), indicating that system failures are not driven by traffic load or peak demand.

2. Failure rates remain consistently flat across hours and days, suggesting reliability issues are systemic and uniform rather than time-dependent or usage-driven.

3. The absence of temporal patterns implies that root causes lie in persistent infrastructure or integration inefficiencies, not in temporal spikes or user behavior cycles.

**Hypothesis Conclusion:** The hypothesis that failures increase during peak hours or vary by weekday is rejected; temporal demand does not significantly influence failure rates.


## 3. Network Impact

**Question:** Does network type affect failure rate?

**Hypothesis:** Lower-quality networks will show higher failure rates than faster connections.

**Raw vs Aggregated Comparison:**
- **Raw chart:** Groups every transaction by network_type column and computes failure rate from individual records
- **Aggregated chart:** Pre-grouped failure rates by network_type from the summary table
- If rates match, network_type coding is consistent between datasets; if they diverge, check for data type mismatches or missing network classifications

In [ ]:
network_raw = summarize_raw(raw_df, "network_type") if "network_type" in raw_df.columns else pd.DataFrame()
network_agg = summarize_agg(agg_df, "network_type") if "network_type" in agg_df.columns else pd.DataFrame()

if len(network_raw):
    fig_network_raw = px.bar(network_raw.sort_values("failure_rate", ascending=False), x="network_type", y="failure_rate", title="Raw Failure Rate by Network Type")
    fig_network_raw.update_yaxes(title="Failure Rate (%)")
    fig_network_raw.show()

if len(network_agg):
    fig_network_agg = px.bar(network_agg.sort_values("failure_rate", ascending=False), x="network_type", y="failure_rate", title="Aggregated Failure Rate by Network Type")
    fig_network_agg.update_yaxes(title="Failure Rate (%)")
    fig_network_agg.show()

if len(network_raw):
    contingency = contingency_from_raw(raw_df, "network_type")
    chi2, p_value = chi_square_test(contingency)
    print(f"Chi-square p-value for network-type difference: {p_value:.6f}")

Chi-square p-value for network-type difference: 0.277817


**Conclusion:** 
1. Failure rates are nearly identical across all network types (3G, 4G, WiFi, 5G), indicating that network quality does not materially impact transaction reliability.

2. The minimal variation between the worst (3G) and best (5G/WiFi) performing networks suggests that failures are not driven by connectivity limitations but by backend or integration issues.

3. Consistency between raw and aggregated results confirms that network-related metrics are reliable and not affected by data inconsistencies.

**Hypothesis Conclusion:** The hypothesis that lower-quality networks lead to higher failure rates is rejected; network type does not significantly influence transaction failure rates.


## 4. Device Impact

**Question:** Do certain devices fail more often?

**Hypothesis:** Device type should show measurable differences in failure rate if platform-specific issues exist.

**Raw vs Aggregated Comparison:**
- **Raw chart:** Shows failure rate for each device_type from all transaction-level records; reveals granular device patterns
- **Aggregated chart:** Pre-computed failure rates by device_type; should match raw closely if device classifications are stable
- Discrepancies suggest either device_type values are coded differently or the aggregation scope is narrower than raw data

In [ ]:
device_raw = summarize_raw(raw_df, "device_type") if "device_type" in raw_df.columns else pd.DataFrame()
device_agg = summarize_agg(agg_df, "device_type") if "device_type" in agg_df.columns else pd.DataFrame()

if len(device_raw):
    fig_device_raw = px.bar(device_raw.sort_values("failure_rate", ascending=False), x="device_type", y="failure_rate", title="Raw Failure Rate by Device Type")
    fig_device_raw.update_yaxes(title="Failure Rate (%)")
    fig_device_raw.show()

if len(device_agg):
    fig_device_agg = px.bar(device_agg.sort_values("failure_rate", ascending=False), x="device_type", y="failure_rate", title="Aggregated Failure Rate by Device Type")
    fig_device_agg.update_yaxes(title="Failure Rate (%)")
    fig_device_agg.show()

if len(device_raw):
    contingency = contingency_from_raw(raw_df, "device_type")
    chi2, p_value = chi_square_test(contingency)
    print(f"Chi-square p-value for device-type difference: {p_value:.6f}")

Chi-square p-value for device-type difference: 0.554019


**Conclusion:**
1. Failure rates are nearly identical across device types (Web, Android, iOS), indicating that platform-specific issues do not significantly impact transaction reliability.

2. The slightly higher failure rate on web compared to mobile is marginal and not operationally significant, suggesting that failures are not driven by frontend or device-level constraints.

3. Strong alignment between raw and aggregated results confirms consistent device classification and reliable measurement across datasets.

**Hypothesis Conclusion:** The hypothesis that device type drives measurable differences in failure rates is rejected; failures are not influenced by platform or device.


## 5. Bank-Level Reliability

**Question:** Which banks contribute most to failures?

**Hypothesis:** Some sender banks will show higher failure rates, and receiver-bank patterns may reveal integration pressure in the raw data.

In [ ]:
sender_bank_raw = summarize_raw(raw_df, "sender_bank") if "sender_bank" in raw_df.columns else pd.DataFrame()
receiver_bank_raw = summarize_raw(raw_df, "receiver_bank") if "receiver_bank" in raw_df.columns else pd.DataFrame()
sender_bank_agg = summarize_agg(agg_df, "sender_bank") if "sender_bank" in agg_df.columns else pd.DataFrame()

if len(sender_bank_raw):
    fig_sender_bank = px.bar(sender_bank_raw.sort_values("failure_rate", ascending=False).head(15), x="sender_bank", y="failure_rate", title="Raw Failure Rate by Sender Bank")
    fig_sender_bank.update_yaxes(title="Failure Rate (%)")
    fig_sender_bank.update_xaxes(tickangle=45)
    fig_sender_bank.show()

if len(sender_bank_agg):
    fig_sender_bank_agg = px.bar(sender_bank_agg.sort_values("failure_rate", ascending=False).head(15), x="sender_bank", y="failure_rate", title="Aggregated Failure Rate by Sender Bank")
    fig_sender_bank_agg.update_yaxes(title="Failure Rate (%)")
    fig_sender_bank_agg.update_xaxes(tickangle=45)
    fig_sender_bank_agg.show()

if len(receiver_bank_raw):
    fig_receiver_bank = px.bar(receiver_bank_raw.sort_values("failure_rate", ascending=False).head(15), x="receiver_bank", y="failure_rate", title="Raw Failure Rate by Receiver Bank")
    fig_receiver_bank.update_yaxes(title="Failure Rate (%)")
    fig_receiver_bank.update_xaxes(tickangle=45)
    fig_receiver_bank.show()

if "sender_bank" in raw_df.columns and "receiver_bank" in raw_df.columns:
    bank_heat = pd.crosstab(raw_df["sender_bank"], raw_df["receiver_bank"])
    fig_bank_heat = px.imshow(bank_heat, aspect="auto", title="Sender Bank x Receiver Bank Transaction Volume")
    fig_bank_heat.show()

**Conclusion:** 
1. Failure rates are consistent across all sender and receiver banks, indicating that no individual bank or bank pair disproportionately contributes to transaction failures, and ruling out integration-specific issues.

2. Despite SBI dominating transaction volume and acting as a central hub, its failure rate remains aligned with other banks, showing that high-load nodes are handled efficiently without degrading reliability.

3. Uniform failure rates across uneven transaction flows suggest that failures are driven by system-wide factors rather than bank-level or pair-specific bottlenecks.

**Hypothesis Conclusion:** The hypothesis that certain banks or bank interactions exhibit higher failure rates is rejected; failures are not bank-specific but systemic across the network.

## 6. Transaction Load vs Failure

**Question:** Do failures increase during high traffic?

**Hypothesis:** Higher transaction volume during certain hours should align with higher failure rates if the system is under load.

**Raw-Only Analysis:**
This analysis uses raw transaction data only (no aggregated equivalent), allowing hour-by-hour and peak-hour detail not available in pre-grouped summaries.

In [ ]:
hour_load = summarize_raw(raw_df, "hour_of_day") if "hour_of_day" in raw_df.columns else pd.DataFrame()

if len(hour_load):
    hour_load = hour_load.sort_values("hour_of_day")
    fig_load = go.Figure()
    fig_load.add_bar(x=hour_load["hour_of_day"], y=hour_load["total_transactions"], name="Total Transactions")
    fig_load.add_scatter(x=hour_load["hour_of_day"], y=hour_load["failure_rate"], name="Failure Rate (%)", yaxis="y2")
    fig_load.update_layout(
        title="Transaction Load vs Failure by Hour",
        yaxis=dict(title="Total Transactions"),
        yaxis2=dict(title="Failure Rate (%)", overlaying="y", side="right"),
    )
    fig_load.show()

if "is_peak_hour" in raw_df.columns:
    peak_load = summarize_raw(raw_df, "is_peak_hour")
    fig_peak_load = px.bar(peak_load, x="is_peak_hour", y="failure_rate", title="Failure Rate by Peak Hour Segment")
    fig_peak_load.update_yaxes(title="Failure Rate (%)")
    fig_peak_load.show()

**Conclusion:**
1. Failure rates remain stable across varying transaction volumes, with no increase during peak hours, indicating that system performance does not degrade under high load and is not capacity-constrained.

2. Slightly higher failure rates observed during off-peak hours (12–5 AM) are minimal and likely driven by low transaction volume effects rather than a true degradation in system reliability.

3. The absence of correlation between transaction load and failure rate suggests that failures are independent of traffic intensity and are likely caused by underlying system or process-level factors.

**Hypothesis Conclusion:** The hypothesis that higher transaction volume leads to increased failure rates is rejected; system load does not significantly influence transaction failures.


## 7. Amount Sensitivity

**Question:** Do high-value transactions fail more often?

**Hypothesis:** Larger payment amounts will have a higher failure rate if stricter checks or risk controls are triggered on expensive transactions.

**Raw vs Aggregated Comparison:**
- **Raw chart:** Groups transactions into amount buckets (small <500, medium 500-2000, high >2000) based on transaction_amount; shows actual payment-size patterns
- **Aggregated chart:** Groups records by avg_amount from pre-aggregated data, then buckets by average; smooths out individual outliers and shows trend at summary level
- Raw buckets show transaction-level sensitivity; aggregated buckets show whether high-average groups align with raw high-transaction-count groups
- If raw small-bucket fails more but aggregated small-average succeeds, investigate whether average-amount aggregation is hiding per-transaction detail

In [ ]:
amount_raw = raw_df.copy()
amount_raw["amount_bucket"] = pd.cut(
    amount_raw["amount_inr"],
    bins=[-np.inf, 500, 2000, np.inf],
    labels=["small", "medium", "high"],
)

amount_summary = summarize_raw(amount_raw, "amount_bucket") if "amount_bucket" in amount_raw.columns else pd.DataFrame()
if len(amount_summary):
    fig_amount = px.bar(amount_summary.sort_values("failure_rate", ascending=False), x="amount_bucket", y="failure_rate", title="Failure Rate by Amount Bucket")
    fig_amount.update_yaxes(title="Failure Rate (%)")
    fig_amount.show()

if "avg_amount" in agg_df.columns:
    agg_amount_data = agg_df.copy()
    agg_amount_data["amount_bucket"] = pd.cut(
        agg_amount_data["avg_amount"],
        bins=[-np.inf, 500, 2000, np.inf],
        labels=["small", "medium", "high"],
    )
    agg_amount_summary = summarize_agg(agg_amount_data, "amount_bucket")
    fig_agg_amount = px.bar(
        agg_amount_summary.sort_values("failure_rate", ascending=False),
        x="amount_bucket",
        y="failure_rate",
        title="Aggregated Failure Rate by Avg Amount Bucket",
    )
    fig_agg_amount.update_yaxes(title="Failure Rate (%)")
    fig_agg_amount.show()

**Conclusion:**
1. Raw (weighted) analysis shows failure rates are nearly identical across small, medium, and high transaction amounts, indicating that transaction value does not materially influence reliability.

2. The higher failure rate observed for small transactions in the aggregated view is a result of unweighted averaging, where low-volume segments with higher variance disproportionately inflate the metric.

3. The discrepancy between raw and aggregated results reflects differences in aggregation methodology rather than data inconsistency, reinforcing that weighted, transaction-level calculations provide the accurate view.

**Hypothesis Conclusion:** The hypothesis that transaction amount influences failure rates is rejected; failures are not dependent on transaction value.

## 8. Geography Impact

**Question:** Are failures region-specific?

**Hypothesis:** Some sender states will show higher failure rates because of local infrastructure or operational differences.

**Raw-Only Analysis:**
Sender state analysis is only available in raw transaction data; the aggregated dataset does not group by geographic location.

In [ ]:
state_raw = summarize_raw(raw_df, "sender_state") if "sender_state" in raw_df.columns else pd.DataFrame()

if len(state_raw):
    fig_state = px.bar(state_raw.sort_values("failure_rate", ascending=False).head(15), x="sender_state", y="failure_rate", title="Raw Failure Rate by Sender State")
    fig_state.update_yaxes(title="Failure Rate (%)")
    fig_state.update_xaxes(tickangle=45)
    fig_state.show()

**Conclusion:**
1. Failure rates are consistently uniform across all sender states, with only negligible variation, indicating that geographic location does not influence transaction reliability.

2. The absence of any high- or low-performing states suggests that regional infrastructure differences or local operational conditions are not contributing to failures.

3. The tight clustering of failure rates across states reinforces that failures are system-wide and not driven by geographic disparities.

**Hypothesis Conclusion:** The hypothesis that certain states exhibit higher failure rates due to regional factors is rejected; transaction failures are not geographically dependent.


## 9. Interaction Effects

**Question:** Which combinations fail most often?

**Hypothesis:** Network-by-time and bank-by-network combinations will expose concentrated failure pockets that single-variable charts can miss.

**Raw-Only Analysis:**
Interaction heatmaps require cross-dimensional grouping (network × hour, bank × network) only possible at transaction-level; not pre-computed in aggregated dataset.

In [ ]:
if {"network_type", "hour_of_day"}.issubset(raw_df.columns):
    network_hour = raw_df.copy()
    network_hour["_failed"] = raw_failure_mask(network_hour).astype(int)
    pivot_network_hour = network_hour.pivot_table(index="network_type", columns="hour_of_day", values="_failed", aggfunc="mean") * 100
    fig_network_hour = px.imshow(pivot_network_hour, aspect="auto", title="Failure Rate Heatmap: Network Type x Hour")
    fig_network_hour.show()

if {"sender_bank", "network_type"}.issubset(raw_df.columns):
    bank_network = raw_df.copy()
    bank_network["_failed"] = raw_failure_mask(bank_network).astype(int)
    pivot_bank_network = bank_network.pivot_table(index="sender_bank", columns="network_type", values="_failed", aggfunc="mean") * 100
    fig_bank_network = px.imshow(pivot_bank_network, aspect="auto", title="Failure Rate Heatmap: Sender Bank x Network Type")
    fig_bank_network.show()

**Conclusion:**
1. Although the network × hour heatmap shows a wider range (~2%–8%), most combinations cluster around the baseline (~5%), indicating that variations are not strong or consistent enough to suggest a meaningful interaction effect.

2. Higher failure rates observed in certain cells (e.g., WiFi during off-hours) are likely driven by low transaction volumes, where small absolute changes inflate percentages rather than reflecting true system degradation.

3. No interaction combination (network × hour or bank × network) consistently exhibits elevated failure rates across high-volume segments, indicating the absence of concentrated failure pockets.

**Hypothesis Conclusion:** The hypothesis that interaction effects reveal significant failure hotspots is rejected; observed variations are minor, inconsistent, and largely attributable to low-volume noise rather than meaningful system behavior.


## 9. Interaction Effects

**Question:** Which combinations fail most often?

**Hypothesis:** Network-by-time and bank-by-network combinations will expose concentrated failure pockets that single-variable charts can miss.

**Raw-Only Analysis:**
Interaction heatmaps require cross-dimensional grouping (network × hour, bank × network) only possible at transaction-level; not pre-computed in aggregated dataset.

In [ ]:
driver_tables = []

for label, summary, group_col in [
    ("hour_of_day", hour_summary, "hour_of_day"),
    ("day_of_week", day_summary, "day_of_week"),
    ("network_type", network_raw, "network_type"),
    ("device_type", device_raw, "device_type"),
    ("sender_bank", sender_bank_raw, "sender_bank"),
    ("receiver_bank", receiver_bank_raw, "receiver_bank"),
    ("sender_state", state_raw, "sender_state"),
]:
    if isinstance(summary, pd.DataFrame) and len(summary):
        top_row = summary.sort_values(["failure_rate", "total_transactions"], ascending=[False, False]).head(1).copy()
        top_row["dimension"] = label
        driver_tables.append(top_row)

if driver_tables:
    driver_df = pd.concat(driver_tables, ignore_index=True, sort=False)
    driver_df = driver_df[[c for c in ["dimension", "hour_of_day", "day_of_week", "network_type", "device_type", "sender_bank", "receiver_bank", "sender_state", "total_transactions", "failed_transactions", "failure_rate"] if c in driver_df.columns]]
    fig_driver = go.Figure(data=[go.Table(
        header=dict(values=list(driver_df.columns), align="left"),
        cells=dict(values=[driver_df[col] for col in driver_df.columns], align="left"),
)])
    fig_driver.update_layout(title="Top Failure Drivers Across Dimensions")
    fig_driver.show()

    top_driver = driver_df.sort_values(["failure_rate", "total_transactions"], ascending=[False, False]).iloc[0]
    print(f"Top driver: {top_driver['dimension']} | failure_rate={top_driver['failure_rate']:.2f}%")

Top driver: hour_of_day | failure_rate=5.40%


**Conclusion:** The synthesis table above identifies the most reliable and least reliable components for the dashboard narrative.